# CUDA Telemetry Engine demo

Compare measurements with reference values using C++20 on the CPU or CUDA on an NVIDIA GPU.
The engine reports sample count, threshold violations, maximum absolute error, and mean squared error.

Select a **T4 GPU** runtime, then run the cells in order. The setup cell creates a new temporary
folder each time it runs. It does not replace an earlier project folder. After a runtime reset,
run setup again. No ZIP uploads are needed.

This notebook uses a fixed source commit. It contains no saved execution results.
The optional benchmark and sanitizer cells are disabled by default.

## 1. Get the source and check the environment

The commit below includes the verified CSV feature and CPU CI workflow. A fixed commit makes
it clear which code a run used. The helper prints each command, records its output, and stops
on failure. The downloaded run log includes the source commit and tool versions.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import shutil
import subprocess
import tempfile

REVISION = "b34ee6e7bd7a9911d39c22b63e11875483bf7137"
work = Path(tempfile.mkdtemp(prefix="telemetry-demo-", dir="/content"))
root = work / "repo"
results = work / "results"
results.mkdir()
log_path = results / "run.log"
log_path.write_text(f"Started UTC: {datetime.now(timezone.utc).isoformat()}\n")

def run(command, cwd=None):
    heading = "Running: " + " ".join(command)
    print("\n" + heading, flush=True)
    with log_path.open("a") as log:
        log.write("\n" + heading + "\n")
        completed = subprocess.run(
            command, cwd=cwd or root,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        print(completed.stdout, flush=True)
        log.write(completed.stdout)
        log.write(f"Exit status: {completed.returncode}\n")
    completed.check_returncode()
    return completed.stdout

run(["git", "clone", "https://github.com/CorbinMilne/cuda-telemetry-engine.git", str(root)], cwd=work)
run(["git", "checkout", "--detach", REVISION])
actual_revision = run(["git", "rev-parse", "HEAD"]).strip()
if actual_revision != REVISION:
    raise RuntimeError("The source commit does not match the requested revision.")
(results / "source-revision.txt").write_text(actual_revision + "\n")
for command in (["nvidia-smi"], ["nvcc", "--version"], ["g++", "--version"],
                ["cmake", "--version"], ["python3", "--version"]):
    run(command)
print(f"Demo folder: {work}")

## 2. Build and test

CMake builds both backends in Release mode. Architecture 75 targets the T4.
Expect five passing CTest suites and five passing Python tests.
The Python tests check the benchmark script without running a benchmark.

In [ ]:
run([
    "cmake", "-S", ".", "-B", "build-cuda",
    "-DCMAKE_BUILD_TYPE=Release", "-DTELEMETRY_ENABLE_CUDA=ON",
    "-DBUILD_TESTING=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",
])
run(["cmake", "--build", "build-cuda", "--parallel", "2"])
run(["ctest", "--test-dir", "build-cuda", "--output-on-failure"])
run(["python3", "-m", "unittest", "discover", "-s", "tests", "-p", "benchmark_tests.py"])

## 3. Validate the example CSV

The example is handwritten test data, not recorded sensor data. Each row contains a measurement
and its expected reference in the same units. Its absolute errors are 0, 0.5, 2, and 4.
At threshold 1, both backends should report:

- Samples: 4
- Violations: 2
- Maximum absolute error: 4
- Mean squared error: (0 + 0.25 + 4 + 16) / 4 = 5.0625

CSV loading happens before validation timing. Four samples check correctness, not useful GPU
performance. The GPU has allocation, transfer, and launch costs even for a tiny input.

In [ ]:
print((root / "data/example.csv").read_text())
for backend in ("cpu", "cuda"):
    run([
        "./build-cuda/telemetry_engine", "--input", "data/example.csv",
        "--backend", backend, "--threshold", "1.0",
    ])

## 4. Optional: benchmark larger batches

Set RUN_BENCHMARKS to True to run the existing sweep. It compares both backends on the same
machine at four sample counts, alternating their execution order over four pairs per size.
Each invocation reports the median of five timed calls after one warm-up.

Complete validation time includes CUDA allocations, copies, kernels, and cleanup. The GPU
kernel interval is reported separately. CPU/CUDA ratios above 1 favor CUDA. Shared runtime
conditions can change results; this run does not guarantee a speedup on other workloads.
The sweep uses synthetic data. Each run saves new raw records without replacing earlier results.

In [ ]:
RUN_BENCHMARKS = False
if RUN_BENCHMARKS:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    run([
        "python3", "benchmarks/run_benchmarks.py", "./build-cuda/telemetry_engine",
        "--output-dir", str(results / f"benchmark-{stamp}"),
    ])
else:
    print("Benchmark skipped. Set RUN_BENCHMARKS = True to enable it.")

## 5. Optional: check GPU memory and synchronization

Set RUN_SANITIZERS to True to run all twelve CUDA cases under each tool.
A separate Release build adds source-line information for error reports.

- memcheck checks invalid memory access and GPU allocation leaks.
- racecheck checks shared-memory access hazards between threads.
- initcheck checks uninitialized global-memory reads.
- synccheck checks synchronization errors.

Expect zero errors, zero leaked bytes, and zero race hazards or warnings. These checks cover
the executed tests, not every possible input. Instrumented runs are not performance measurements.
See the [NVIDIA guide](https://docs.nvidia.com/compute-sanitizer/ComputeSanitizer/index.html).

In [ ]:
RUN_SANITIZERS = False
if RUN_SANITIZERS:
    sanitizer = shutil.which("compute-sanitizer")
    if sanitizer is None:
        raise RuntimeError("compute-sanitizer is not available in this runtime.")
    run([sanitizer, "--version"])
    run([
        "cmake", "-S", ".", "-B", "build-sanitizer",
        "-DCMAKE_BUILD_TYPE=Release", "-DTELEMETRY_ENABLE_CUDA=ON",
        "-DBUILD_TESTING=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",
        "-DCMAKE_CUDA_FLAGS=-lineinfo",
    ])
    run(["cmake", "--build", "build-sanitizer", "--target", "telemetry_cuda_tests", "--parallel", "2"])
    for tool in ("memcheck", "racecheck", "initcheck", "synccheck"):
        command = [sanitizer, "--tool", tool, "--error-exitcode", "1"]
        if tool == "memcheck":
            command += ["--leak-check", "full"]
        run(command + ["./build-sanitizer/telemetry_cuda_tests"])
else:
    print("Sanitizers skipped. Set RUN_SANITIZERS = True to enable them.")

## 6. Save the run records

Download the log, source revision, and any benchmark records before leaving the runtime.
You can run this cell after a failed command to preserve its diagnostic output, or rerun it
if you later enable an optional check. A successful ZIP download alone does not mean the tests passed.

In [ ]:
from google.colab import files

archive = shutil.make_archive(str(work / "telemetry-demo-results"), "zip", root_dir=results)
files.download(archive)